## 1. Contexto y carga de datos

El objetivo es estimar el ingreso comercial de la **siguiente jornada con transacciones registradas**, para apoyar la planificación comercial. Fuente de trabajo: `Online Retail.xlsx` (importe en libras esterlinas). La lectura requiere que el Excel esté en el directorio desde el que se ejecuta el notebook. Los registros corresponden a líneas de transacciones, no a ventas diarias previamente agregadas.


In [ ]:

import pandas as pd

# Cargar los datos de ventas
df = pd.read_excel("Online Retail.xlsx")

# Mostrar las primeras cinco filas
display(df.head())

# Consultar el tamaño del dataset
print("Filas y columnas:", df.shape)

# Comprobar las columnas y sus tipos
df.info()


## 2. Auditoría inicial de calidad

Se cuantifican los valores faltantes, duplicados, cancelaciones, cantidades y precios no positivos. No se eliminan automáticamente los duplicados ni los `CustomerID` faltantes: requieren interpretación y su supresión alteraría los resultados.


In [ ]:

# Comprobamos los valores ausentes
print("VALORES AUSENTES:")
print(df.isna().sum())

# Comprobamos los registros duplicados
print("\nFILAS DUPLICADAS:")
print(df.duplicated().sum())

# Comprobamos las fechas disponibles
print("\nRANGO DE FECHAS:")
print("Primera fecha:", df["InvoiceDate"].min())
print("Última fecha:", df["InvoiceDate"].max())


In [ ]:

# Facturas identificadas como cancelaciones
print("CANCELACIONES:")
print(df["InvoiceNo"].astype(str).str.startswith("C").sum())

# Cantidades negativas o cero
print("\nCANTIDADES NEGATIVAS O CERO:")
print((df["Quantity"] <= 0).sum())

# Precios negativos o cero
print("\nPRECIOS NEGATIVOS O CERO:")
print((df["UnitPrice"] <= 0).sum())

# Número de días con registros
print("\nDÍAS CON TRANSACCIONES:")
print(df["InvoiceDate"].dt.normalize().nunique())


In [ ]:

# Identificar condiciones que queremos investigar
cancelacion = df["InvoiceNo"].astype(str).str.startswith("C")
cantidad_no_positiva = df["Quantity"] <= 0
precio_no_positivo = df["UnitPrice"] <= 0

# Comprobar si los registros se solapan
print("Cancelación y cantidad no positiva:",
      (cancelacion & cantidad_no_positiva).sum())

print("Cantidad y precio no positivos:",
      (cantidad_no_positiva & precio_no_positivo).sum())

print("Al menos una condición:",
      (cancelacion | cantidad_no_positiva | precio_no_positivo).sum())

# Mostrar ejemplos de precios no positivos
print("\nEjemplos con precio no positivo:")
display(df.loc[
    precio_no_positivo,
    ["InvoiceNo", "Description", "Quantity", "UnitPrice"]
].head(10))

# Comprobar días sin transacciones
fechas = pd.date_range(
    df["InvoiceDate"].min().normalize(),
    df["InvoiceDate"].max().normalize(),
    freq="D"
)

dias_presentes = pd.DatetimeIndex(
    df["InvoiceDate"].dt.normalize().unique()
)

print("\nDías del periodo sin registros:",
      len(fechas.difference(dias_presentes)))


In [ ]:

# Identificamos las cancelaciones
cancelacion = df["InvoiceNo"].astype(str).str.startswith("C")

# Cantidades negativas sin factura de cancelación
negativa_sin_c = (df["Quantity"] < 0) & (~cancelacion)

print("Cantidades negativas sin prefijo C:",
      negativa_sin_c.sum())

# Diferenciamos precios cero y negativos
print("Precios exactamente cero:",
      (df["UnitPrice"] == 0).sum())

print("Precios negativos:",
      (df["UnitPrice"] < 0).sum())

# Mostramos ejemplos
print("\nEjemplos de cantidades negativas sin C:")

display(df.loc[
    negativa_sin_c,
    ["InvoiceNo", "Description", "Quantity", "UnitPrice"]
].head(10))

# Distribución de los días sin registros
fechas_faltantes = fechas.difference(dias_presentes)

print("\nDías sin registros por día de la semana:")

print(pd.Series(
    fechas_faltantes.day_name()
).value_counts())


In [ ]:

# Examinar los registros con precio negativo
print("REGISTROS CON PRECIO NEGATIVO:")

display(df.loc[
    df["UnitPrice"] < 0,
    ["InvoiceNo", "StockCode", "Description",
     "Quantity", "UnitPrice", "InvoiceDate"]
])

# Definir una condición provisional para ventas
ventas_candidatas = (
    ~df["InvoiceNo"].astype(str).str.startswith("C")
    & (df["Quantity"] > 0)
    & (df["UnitPrice"] > 0)
)

print("\nLÍNEAS CANDIDATAS A VENTAS:")
print(ventas_candidatas.sum())

print("\nLÍNEAS QUE NO CUMPLEN EL CRITERIO:")
print((~ventas_candidatas).sum())


## 3. Primer escenario: ventas positivas y duplicados

Se construye una vista preliminar de importes positivos. Es exploratoria y **no** representa todavía el indicador comercial final, porque ignora ajustes y cancelaciones. La evaluación de duplicados considera que varias líneas aparentemente iguales podrían ser transacciones legítimas.


In [ ]:

# Crear una copia sin modificar los datos originales
ventas_df = df.loc[ventas_candidatas].copy()

# Calcular el ingreso de cada línea
ventas_df["IngresoLinea"] = (
    ventas_df["Quantity"] * ventas_df["UnitPrice"]
)

print("Líneas candidatas:", len(ventas_df))

print("Ingresos totales provisionales:",
      round(ventas_df["IngresoLinea"].sum(), 2))

print("Productos diferentes:",
      ventas_df["StockCode"].nunique())

print("Días con ventas candidatas:",
      ventas_df["InvoiceDate"].dt.normalize().nunique())

# Mostrar los primeros resultados
display(ventas_df[
    ["InvoiceDate", "Description",
     "Quantity", "UnitPrice", "IngresoLinea"]
].head())


In [ ]:

# PASO 8: COMPROBAR DUPLICADOS EN VENTAS

# Utilizamos las 8 columnas originales
columnas_originales = df.columns.tolist()

# Contamos las filas completamente repetidas
duplicados_ventas = ventas_df.duplicated(
    subset=columnas_originales
).sum()

print("Duplicados en ventas candidatas:",
      duplicados_ventas)

# Comprobamos descripciones ausentes
print("Descripciones vacías:",
      ventas_df["Description"].isna().sum())

# Mostramos ejemplos de filas repetidas
display(
    ventas_df.loc[
        ventas_df.duplicated(
            subset=columnas_originales,
            keep=False
        )
    ].head(10)
)


In [ ]:

# PASO 9: ANALISIS DE DUPLICADOS

columnas_originales = df.columns.tolist()

duplicados = ventas_df.duplicated(
    subset=columnas_originales
)

importe_duplicados = ventas_df.loc[
    duplicados, "IngresoLinea"
].sum()

ingresos_actuales = ventas_df["IngresoLinea"].sum()

print("Importe asociado a duplicados:",
      round(importe_duplicados, 2))

print("Ingresos conservando duplicados:",
      round(ingresos_actuales, 2))

print("Ingresos sin duplicados:",
      round(ingresos_actuales - importe_duplicados, 2))

print("Diferencia porcentual:",
      round(importe_duplicados / ingresos_actuales * 100, 2), "%")


# AGRUPACION PROVISIONAL POR FECHA

ventas_df["Fecha"] = ventas_df["InvoiceDate"].dt.normalize()

ventas_diarias = (
    ventas_df.groupby("Fecha", as_index=False)["IngresoLinea"]
    .sum()
    .rename(columns={"IngresoLinea": "IngresosDiarios"})
    .sort_values("Fecha")
)

print("\nPrimeros ingresos diarios:")
display(ventas_diarias.head())

print("\nEstadisticas descriptivas:")
display(ventas_diarias["IngresosDiarios"].describe())


## 4. Calendario y comportamiento de los ingresos

Se contrasta el período calendario con las fechas que registran actividad. **Los días sin registro no se convierten en ceros**: no puede confirmarse si corresponden a cierres, ausencia de operaciones o información no disponible.


In [ ]:

# Fechas del periodo completo
calendario = pd.date_range(
    df["InvoiceDate"].min().normalize(),
    df["InvoiceDate"].max().normalize(),
    freq="D"
)

# Fechas del archivo original
dias_originales = pd.DatetimeIndex(
    df["InvoiceDate"].dt.normalize().unique()
)

# Fechas con ventas candidatas
dias_ventas = pd.DatetimeIndex(
    ventas_df["InvoiceDate"].dt.normalize().unique()
)

# Días sin registros
sin_movimientos = calendario.difference(dias_originales)

# Días con registros, pero sin ventas candidatas
sin_ventas = dias_originales.difference(dias_ventas)

print("Días del calendario:", len(calendario))
print("Días con registros:", len(dias_originales))
print("Días con ventas:", len(dias_ventas))
print("Días sin registros:", len(sin_movimientos))
print("Días solo con otros movimientos:", len(sin_ventas))

# Comprobación de coherencia
print("Control calendario:",
      len(calendario) == len(dias_originales) + len(sin_movimientos))

print("Control ventas:",
      len(dias_originales) == len(dias_ventas) + len(sin_ventas))


In [ ]:

import matplotlib.pyplot as plt

# Comprobamos que mantenemos los datos originales
print("Filas originales:", len(df))

# Incorporamos todas las fechas del calendario
# Los días sin registros quedan como NaN
serie_calendario = (
    ventas_diarias.set_index("Fecha")["IngresosDiarios"]
    .reindex(calendario)
)

print("Días sin registros:",
      serie_calendario.isna().sum())

# Dibujamos los ingresos diarios
plt.figure(figsize=(12, 5))

plt.plot(
    serie_calendario.index,
    serie_calendario.values
)

plt.title("Evolución de ingresos diarios - Online Retail")
plt.xlabel("Fecha")
plt.ylabel("Ingresos diarios provisionales (£)")
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

# Guardamos el gráfico para el TFM
plt.savefig("evolucion_ingresos_diarios.png", dpi=150)

plt.show()


In [ ]:

# PASO 12: ANALIZAR LOS INGRESOS EXTRAORDINARIOS

# Seleccionamos los diez días con más ingresos
top_dias = ventas_diarias.nlargest(
    10, "IngresosDiarios"
)

print("LOS 10 DÍAS CON MAYORES INGRESOS:")
display(top_dias)

# Identificamos el día con mayores ingresos
fecha_max = top_dias.iloc[0]["Fecha"]

# Seleccionamos sus transacciones
detalle_max = ventas_df[
    ventas_df["Fecha"] == fecha_max
]

print("\nFecha de ingresos máximos:", fecha_max)
print("Número de facturas:", detalle_max["InvoiceNo"].nunique())
print("Número de líneas:", len(detalle_max))

# Mostramos las cinco líneas con mayor importe
print("\nOPERACIONES DE MAYOR IMPORTE:")

display(
    detalle_max.nlargest(5, "IngresoLinea")[
        ["InvoiceNo", "Description",
         "Quantity", "UnitPrice", "IngresoLinea"]
    ]
)


## 5. Investigación de valores extremos

Se estudia una operación extraordinaria y su cancelación posterior, especialmente el cierre de la serie. El último día del archivo contiene actividad solo hasta las 12:50 y se excluye del experimento predictivo por su posible carácter incompleto.


In [ ]:

# PASO 13: ANALISIS DE SENSIBILIDAD

# Identificamos la operación extraordinaria
operacion_extrema = (
    (ventas_df["InvoiceNo"].astype(str) == "581483")
    & (ventas_df["StockCode"].astype(str) == "23843")
    & (ventas_df["Quantity"] == 80995)
)

# Revisamos cuántas líneas cumplen la condición
print("Líneas encontradas:", operacion_extrema.sum())

# Calculamos el impacto
importe_extremo = ventas_df.loc[
    operacion_extrema, "IngresoLinea"
].sum()

ingresos_dia = ventas_diarias.loc[
    ventas_diarias["Fecha"] == pd.Timestamp("2011-12-09"),
    "IngresosDiarios"
].iloc[0]

print("Ingresos originales del día:", round(ingresos_dia, 2))
print("Importe operación:", round(importe_extremo, 2))
print("Ingresos sin la operación:",
      round(ingresos_dia - importe_extremo, 2))

print("Porcentaje que representa:",
      round(importe_extremo / ingresos_dia * 100, 2), "%")


In [ ]:

# PASO 14: INVESTIGAR POSIBLE DEVOLUCION

# Buscamos el producto de la operación extraordinaria
producto = df[
    df["StockCode"].astype(str) == "23843"
].copy()

# Calculamos el importe con signo
producto["Importe"] = (
    producto["Quantity"] * producto["UnitPrice"]
)

print("OPERACIONES DEL PRODUCTO 23843:")

display(
    producto[
        ["InvoiceNo", "InvoiceDate", "Description",
         "Quantity", "UnitPrice", "Importe"]
    ].sort_values("InvoiceDate")
)

# Buscamos devoluciones por 80.995 unidades
devoluciones = producto[
    producto["Quantity"] == -80995
]

print("\nPOSIBLES DEVOLUCIONES:")
display(devoluciones)

print("\nNÚMERO DE DEVOLUCIONES ENCONTRADAS:",
      len(devoluciones))


## 6. Definición de ingresos comerciales y conciliación

Se investigan importes firmados, cancelaciones, descuentos y conceptos administrativos. **Atención:** en las celdas de auditoría se utiliza una exclusión preliminar del código `D`; la **definición operativa final** se fija después y **sí conserva descuentos `D`**, excluyendo `AMAZONFEE`, `BANK CHARGES` y `CRUK`. Los importes calculados son una aproximación de ingresos comerciales a partir de este dataset, no cifras contables auditadas.


In [ ]:

# PASO 15: COMPARAR INGRESOS BRUTOS Y NETOS

# Creamos una copia del dataset original
datos_netos = df.copy()

# Excluimos los precios no positivos
# para no mezclar ajustes contables especiales
datos_netos = datos_netos[
    datos_netos["UnitPrice"] > 0
].copy()

# Calculamos importes con signo:
# ventas positivas y cancelaciones negativas
datos_netos["ImporteNeto"] = (
    datos_netos["Quantity"] * datos_netos["UnitPrice"]
)

# Extraemos la fecha
datos_netos["Fecha"] = (
    datos_netos["InvoiceDate"].dt.normalize()
)

# Calculamos los ingresos netos diarios
netos_diarios = (
    datos_netos.groupby("Fecha")["ImporteNeto"]
    .sum()
    .reset_index()
)

# Comparamos con los ingresos brutos
comparacion = ventas_diarias.merge(
    netos_diarios,
    on="Fecha",
    how="outer"
).sort_values("Fecha")

print("COMPARACIÓN DEL 9 DE DICIEMBRE:")

display(
    comparacion[
        comparacion["Fecha"] == "2011-12-09"
    ]
)

print("\nINGRESOS TOTALES:")
print("Brutos:",
      round(ventas_diarias["IngresosDiarios"].sum(), 2))

print("Netos:",
      round(netos_diarios["ImporteNeto"].sum(), 2))


In [ ]:

# PASO 16: ANALISIS DE INGRESOS NETOS DIARIOS

print("Dias con ingresos positivos:",
      (netos_diarios["ImporteNeto"] > 0).sum())

print("Dias con ingresos negativos:",
      (netos_diarios["ImporteNeto"] < 0).sum())

print("Dias con ingresos cero:",
      (netos_diarios["ImporteNeto"] == 0).sum())

print("\nESTADISTICAS DE INGRESOS NETOS:")
display(netos_diarios["ImporteNeto"].describe())

print("\nLOS 5 DIAS CON MENORES INGRESOS NETOS:")
display(netos_diarios.nsmallest(5, "ImporteNeto"))


In [ ]:

# PASO 17: INVESTIGAR EL DIA NEGATIVO

fecha_negativa = pd.Timestamp("2011-01-05")

movimientos_dia = datos_netos.loc[
    datos_netos["Fecha"] == fecha_negativa
].copy()

# Separamos importes positivos y negativos
positivos = movimientos_dia.loc[
    movimientos_dia["ImporteNeto"] > 0
]

negativos = movimientos_dia.loc[
    movimientos_dia["ImporteNeto"] < 0
]

print("Facturas distintas:",
      movimientos_dia["InvoiceNo"].nunique())

print("Total de ventas positivas:",
      round(positivos["ImporteNeto"].sum(), 2))

print("Total de movimientos negativos:",
      round(negativos["ImporteNeto"].sum(), 2))

print("Saldo neto del día:",
      round(movimientos_dia["ImporteNeto"].sum(), 2))

print("\nCINCO MOVIMIENTOS NEGATIVOS DE MAYOR IMPORTE:")

display(
    negativos.nsmallest(5, "ImporteNeto")[
        ["InvoiceNo", "Description",
         "Quantity", "UnitPrice", "ImporteNeto"]
    ]
)


In [ ]:

# PASO 18: INVESTIGAR OPERACIONES ESPECIALES

# Buscamos registros relacionados con Amazon
amazon = df.loc[
    df["Description"]
    .fillna("")
    .astype(str)
    .str.contains("AMAZON FEE", case=False, regex=False, na=False)
].copy()

# Calculamos su importe
amazon["Importe"] = (
    amazon["Quantity"] * amazon["UnitPrice"]
)

print("REGISTROS AMAZON FEE:")
print("Número de registros:", len(amazon))
print("Importe total:", round(amazon["Importe"].sum(), 2))

display(
    amazon[
        ["InvoiceNo", "StockCode", "Description",
         "Quantity", "UnitPrice", "Importe"]
    ].head(10)
)

# Identificamos códigos con letras u otros símbolos
codigos_especiales = df.loc[
    ~df["StockCode"]
    .fillna("")
    .astype(str)
    .str.fullmatch(r"\d+", na=False)
]

print("\nCÓDIGOS NO EXCLUSIVAMENTE NUMÉRICOS:")

display(
    codigos_especiales["StockCode"]
    .value_counts().head(20)
)


In [ ]:

# PASO 19: PREPARACION DE DATOS COMERCIALES

# Conservamos el dataset original intacto
datos_comerciales = df.copy()

# Identificamos codigos administrativos conocidos
codigos_excluir = [
    "AMAZONFEE",
    "BANK CHARGES",
    "CRUK",
    "D"
]

# Creamos un filtro de operaciones candidatas
filtro = (
    (datos_comerciales["UnitPrice"] > 0)
    & (~datos_comerciales["StockCode"]
       .astype(str).isin(codigos_excluir))
)

# Aplicamos el filtro en una nueva copia
datos_comerciales = datos_comerciales.loc[filtro].copy()

# Calculamos importes conservando el signo
datos_comerciales["ImporteComercial"] = (
    datos_comerciales["Quantity"]
    * datos_comerciales["UnitPrice"]
)

# Extraemos la fecha
datos_comerciales["Fecha"] = (
    datos_comerciales["InvoiceDate"].dt.normalize()
)

# Agrupamos por fecha
ingresos_comerciales = (
    datos_comerciales.groupby("Fecha", as_index=False)
    ["ImporteComercial"]
    .sum()
    .sort_values("Fecha")
)

print("Registros comerciales provisionales:",
      len(datos_comerciales))

print("Ingresos comerciales provisionales:",
      round(datos_comerciales["ImporteComercial"].sum(), 2))

print("Días con registros:",
      len(ingresos_comerciales))

print("\nESTADÍSTICAS:")
display(ingresos_comerciales["ImporteComercial"].describe())

print("\nCINCO DÍAS CON MENORES INGRESOS:")
display(ingresos_comerciales.nsmallest(5, "ImporteComercial"))



In [ ]:

# PASO 20: AUDITORIA FINAL DE EXCLUSIONES

codigos_excluir = [
    "AMAZONFEE", "BANK CHARGES", "CRUK", "D"
]

# Seleccionamos los códigos excluidos
revision = df.loc[
    df["StockCode"].astype(str).isin(codigos_excluir)
].copy()

revision["Importe"] = (
    revision["Quantity"] * revision["UnitPrice"]
)

resumen = (
    revision.groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        Registros=("InvoiceNo", "size"),
        Importe=("Importe", "sum")
    )
    .reset_index()
    .sort_values("Registros", ascending=False)
)

print("CONCEPTOS EXCLUIDOS:")
display(resumen)

print("\nTOTAL EXCLUIDO POR ESTOS CÓDIGOS:")
print(round(revision["Importe"].sum(), 2))

print("\nFECHA FINAL DEL DATASET:")
print(df["InvoiceDate"].max())



## 7. Construcción definitiva de la serie diaria

La serie final retiene líneas con `UnitPrice > 0`, preserva el signo de las cantidades (incluidas cancelaciones), conserva `D` y omite únicamente los tres códigos administrativos documentados. Los importes se agregan por día. La base sigue incluyendo filas duplicadas cuya naturaleza no puede establecerse con certeza.


In [ ]:

# PASO 21: CREAR DATASET COMERCIAL

# Códigos de gastos administrativos
codigos_gastos = [
    "AMAZONFEE",
    "BANK CHARGES",
    "CRUK"
]

# Creamos copia de los datos originales
datos_finales = df.copy()

# Filtramos únicamente gastos conocidos
# y registros sin precio positivo
filtro = (
    (datos_finales["UnitPrice"] > 0)
    & (~datos_finales["StockCode"]
       .astype(str).isin(codigos_gastos))
)

datos_finales = datos_finales.loc[filtro].copy()

# Calculamos ingresos comerciales con signo
datos_finales["IngresoComercial"] = (
    datos_finales["Quantity"]
    * datos_finales["UnitPrice"]
)

# Extraemos la fecha
datos_finales["Fecha"] = (
    datos_finales["InvoiceDate"].dt.normalize()
)

# Agrupamos ingresos por día
serie_final = (
    datos_finales.groupby("Fecha", as_index=False)
    ["IngresoComercial"]
    .sum()
    .sort_values("Fecha")
)

print("Registros seleccionados:", len(datos_finales))

print("Ingresos comerciales:",
      round(datos_finales["IngresoComercial"].sum(), 2))

print("Días disponibles:", len(serie_final))

print("\nESTADÍSTICAS:")
display(serie_final["IngresoComercial"].describe())

print("\nPRIMEROS REGISTROS:")
display(serie_final.head())


## 8. Preparación temporal y exclusión del último día

Para el modelado se excluye el 09/12/2011, jornada excepcional que termina a las 12:50 y puede estar incompleta. Esta decisión se toma como criterio de calidad del dato, no para mejorar artificialmente el resultado predictivo.


In [ ]:

# PASO 22: PREPARAR LA SERIE TEMPORAL

# Creamos una copia para no modificar serie_final
serie_modelo = serie_final.copy()

# Nos aseguramos de que las fechas sean correctas
serie_modelo["Fecha"] = pd.to_datetime(
    serie_modelo["Fecha"]
)

# Ordenamos cronológicamente
serie_modelo = serie_modelo.sort_values(
    "Fecha"
).reset_index(drop=True)

# Excluimos provisionalmente el último día
ultima_fecha = serie_modelo["Fecha"].max()

serie_modelo = serie_modelo.loc[
    serie_modelo["Fecha"] < ultima_fecha
].copy()

# Comprobamos el resultado
print("Última fecha excluida:", ultima_fecha)

print("Días disponibles para modelado:",
      len(serie_modelo))

print("Primera fecha:",
      serie_modelo["Fecha"].min())

print("Última fecha utilizada:",
      serie_modelo["Fecha"].max())

print("\nÚLTIMOS CINCO REGISTROS:")
display(serie_modelo.tail())

print("\nVALORES AUSENTES:")
print(serie_modelo.isna().sum())


## 9. Ingeniería de variables sin mirar al futuro

Se usan día de semana, mes, ingresos de las observaciones anteriores 1, 2 y 7 y promedio de las **siete jornadas registradas anteriores**. `shift(1)` en la media móvil evita utilizar el ingreso del día que se intenta predecir. Las observaciones previas pueden no corresponder a días naturales consecutivos.


In [ ]:

# PASO 23: PREPARAR VARIABLES PARA MACHINE LEARNING

import pandas as pd
import numpy as np

# Partimos de la serie preparada en el paso 22
modelo_df = serie_modelo.copy()

# Ordenamos los registros
modelo_df = modelo_df.sort_values(
    "Fecha"
).reset_index(drop=True)

# Variable objetivo: ingresos del día observado
modelo_df["Objetivo"] = modelo_df["IngresoComercial"]

# Variables de calendario
modelo_df["DiaSemana"] = modelo_df["Fecha"].dt.dayofweek
modelo_df["Mes"] = modelo_df["Fecha"].dt.month

# Ingresos de días anteriores con actividad registrada
modelo_df["IngresoAnterior1"] = modelo_df["Objetivo"].shift(1)
modelo_df["IngresoAnterior2"] = modelo_df["Objetivo"].shift(2)
modelo_df["IngresoAnterior7"] = modelo_df["Objetivo"].shift(7)

# Media de las 7 observaciones anteriores
modelo_df["MediaAnterior7"] = (
    modelo_df["Objetivo"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

# Eliminamos únicamente las primeras filas sin histórico
modelo_df = modelo_df.dropna().reset_index(drop=True)

# Verificamos el resultado
print("FILAS DISPONIBLES:", len(modelo_df))

print("\nVALORES AUSENTES:")
print(modelo_df.isna().sum())

print("\nPRIMEROS REGISTROS:")
display(modelo_df.head())


## 10. Partición cronológica

La muestra se divide en entrenamiento (70 %), validación (15 %) y prueba final (15 %) **sin mezclar fechas**. El conjunto de validación se usa para comparar enfoques; el de prueba se reserva para la estimación final, sin ajustar el modelo tras observarlo. El experimento supone predicciones de un paso por jornada registrada con historial real actualizado.


In [ ]:

# PASO 24: DIVISION CRONOLOGICA

# Variables predictoras
variables = [
    "DiaSemana",
    "Mes",
    "IngresoAnterior1",
    "IngresoAnterior2",
    "IngresoAnterior7",
    "MediaAnterior7"
]

X = modelo_df[variables]
y = modelo_df["Objetivo"]

# Dividimos cronológicamente: 70 % / 15 % / 15 %
n = len(modelo_df)
fin_train = int(n * 0.70)
fin_val = int(n * 0.85)

X_train = X.iloc[:fin_train]
X_val = X.iloc[fin_train:fin_val]
X_test = X.iloc[fin_val:]

y_train = y.iloc[:fin_train]
y_val = y.iloc[fin_train:fin_val]
y_test = y.iloc[fin_val:]

print("ENTRENAMIENTO:", len(X_train), "observaciones")
print("VALIDACIÓN:", len(X_val), "observaciones")
print("PRUEBA:", len(X_test), "observaciones")

print("\nFECHAS DE CADA BLOQUE:")

for nombre, indices in [
    ("Train", X_train.index),
    ("Validación", X_val.index),
    ("Test", X_test.index)
]:
    print(
        nombre,
        modelo_df.loc[indices, "Fecha"].min(),
        "hasta",
        modelo_df.loc[indices, "Fecha"].max()
    )


## 11. Modelos de referencia

Se comparan dos pronósticos sencillos: repetir el ingreso de la última jornada y emplear el promedio de las siete jornadas anteriores. Sus resultados sirven de referencia obligatoria para interpretar si los modelos más complejos aportan valor.


In [ ]:

# PASO 25: MODELOS DE REFERENCIA

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Predicciones usando únicamente información anterior
pred_anterior = X_val["IngresoAnterior1"]
pred_media7 = X_val["MediaAnterior7"]

# Función para evaluar los modelos
def evaluar_modelo(nombre, reales, predicciones):
    mae = mean_absolute_error(reales, predicciones)
    rmse = np.sqrt(
        mean_squared_error(reales, predicciones)
    )

    return {
        "Modelo": nombre,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2)
    }

# Comparación
resultados_base = pd.DataFrame([
    evaluar_modelo(
        "Última jornada",
        y_val,
        pred_anterior
    ),
    evaluar_modelo(
        "Media de 7 jornadas",
        y_val,
        pred_media7
    )
])

print("RESULTADOS DE VALIDACIÓN:")
display(resultados_base)

print("\nMEJOR MODELO SEGÚN MAE:")
display(resultados_base.nsmallest(1, "MAE"))


## 12. Modelos de aprendizaje automático

Se entrenan regresión lineal y Random Forest únicamente con entrenamiento; se evalúan en validación mediante MAE y RMSE. No se realiza ajuste de hiperparámetros utilizando el conjunto de prueba.


In [ ]:

# PASO 26: ENTRENAR Y COMPARAR MODELOS

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Modelo 1: Regresión lineal
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train, y_train)

# Modelo 2: Random Forest
modelo_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

# Predicciones de validación
pred_lineal = modelo_lineal.predict(X_val)
pred_rf = modelo_rf.predict(X_val)

# Evaluación de los dos modelos
resultados_ml = pd.DataFrame([
    evaluar_modelo(
        "Regresión lineal", y_val, pred_lineal
    ),
    evaluar_modelo(
        "Random Forest", y_val, pred_rf
    )
])

# Comparación con los modelos de referencia
comparacion_modelos = pd.concat(
    [resultados_base, resultados_ml],
    ignore_index=True
).sort_values("MAE")

print("COMPARACIÓN DE LOS CUATRO MODELOS:")
display(comparacion_modelos)

print("\nMEJOR MODELO SEGÚN MAE:")
display(comparacion_modelos.head(1))


## 13. Experimento adicional: red neuronal MLP

Se incorpora una MLP como quinto enfoque. Incluye estandarización dentro de un `Pipeline`. Es un experimento complementario: más complejidad no implica mejor generalización. La optimización puede tardar y emitir aviso de no convergencia.


In [ ]:

# PASO 27: RED NEURONAL MLP

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Construimos una red neuronal pequeña
modelo_mlp = Pipeline([
    ("escalado", StandardScaler()),
    ("red_neuronal", MLPRegressor(
        hidden_layer_sizes=(16, 8),
        activation="relu",
        solver="adam",
        alpha=10.0,
        learning_rate_init=0.001,
        max_iter=2000,
        early_stopping=False,
        random_state=42
    ))
])

# Entrenamos usando únicamente entrenamiento
modelo_mlp.fit(X_train, y_train)

# Predecimos sobre validación
pred_mlp = modelo_mlp.predict(X_val)

# Evaluamos
resultado_mlp = evaluar_modelo(
    "Red neuronal MLP",
    y_val,
    pred_mlp
)

print("RESULTADO DE LA RED NEURONAL:")
display(pd.DataFrame([resultado_mlp]))

# Comparamos con los cuatro modelos anteriores
comparacion_completa = pd.concat(
    [
        comparacion_modelos,
        pd.DataFrame([resultado_mlp])
    ],
    ignore_index=True
).sort_values("MAE").reset_index(drop=True)

print("\nCOMPARACIÓN DE LOS CINCO MODELOS:")
display(comparacion_completa)

# Comprobamos si el entrenamiento convergió
red = modelo_mlp.named_steps["red_neuronal"]

print("\nIteraciones utilizadas:", red.n_iter_)
print("Pérdida final de entrenamiento:", red.loss_)


## 14. Explicabilidad global

Se examina la importancia de las variables del Random Forest mediante medidas internas y permutación sobre validación. Estas importancias explican el comportamiento predictivo del modelo, **no relaciones causales**.


In [ ]:

# PASO 28: EXPLICABILIDAD DE RANDOM FOREST

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# 1. Importancia interna del modelo
importancia_interna = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo_rf.feature_importances_
}).sort_values("Importancia", ascending=False)

print("IMPORTANCIA INTERNA DE RANDOM FOREST:")
display(importancia_interna)

# 2. Importancia por permutación
# Utilizamos VALIDACIÓN, no el conjunto de prueba
resultado_perm = permutation_importance(
    modelo_rf,
    X_val,
    y_val,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importancia_permutacion = pd.DataFrame({
    "Variable": X_val.columns,
    "AumentoMAE": resultado_perm.importances_mean,
    "Desviacion": resultado_perm.importances_std
}).sort_values("AumentoMAE", ascending=False)

print("\nIMPORTANCIA POR PERMUTACIÓN:")
display(importancia_permutacion)

# 3. Gráfico de importancia por permutación
plt.figure(figsize=(10, 5))

plt.barh(
    importancia_permutacion["Variable"],
    importancia_permutacion["AumentoMAE"]
)

plt.xlabel("Aumento medio del MAE (£)")
plt.ylabel("Variable")
plt.title("Importancia de variables - Random Forest")

plt.axvline(0, color="black", linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()

plt.savefig(
    "importancia_variables_random_forest.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


## 15. Explicabilidad local con SHAP

SHAP descompone una predicción concreta respecto a un valor de referencia. La explicación es local para la observación elegida y no debe generalizarse a todas las jornadas.


In [ ]:

# PASO 29: EXPLICABILIDAD LOCAL CON SHAP

import shap
import pandas as pd
import matplotlib.pyplot as plt

# Creamos el explicador de Random Forest
explainer = shap.TreeExplainer(modelo_rf)

# Seleccionamos cinco observaciones de validación
X_ejemplos = X_val.iloc[:5].copy()

# Calculamos las contribuciones SHAP
valores_shap = explainer(X_ejemplos)

# Mostramos las variables y sus contribuciones
print("PREDICCIONES ANALIZADAS:")
display(X_ejemplos)

# Explicamos la primera predicción
shap.plots.waterfall(
    valores_shap[0],
    max_display=10,
    show=False
)

plt.tight_layout()
plt.savefig(
    "shap_prediccion_individual.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()


## 16. Evaluación final en prueba

Se contrastan la media de siete jornadas y Random Forest sobre el mismo bloque de 45 jornadas reservadas. No se reentrenan ni retocan a partir de esta evaluación. Resultado registrado en el proyecto: MAE Media7 **14.015,88 £**, MAE Random Forest **18.924,47 £**; estos importes deben comprobarse al ejecutar este notebook en el entorno de entrega.


In [ ]:

# PASO 30: EVALUACION FINAL EN TEST

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predicciones con modelos ya seleccionados
# No reentrenamos ni modificamos hiperparámetros
pred_test_rf = modelo_rf.predict(X_test)

# Referencia: media de las 7 jornadas anteriores
pred_test_base = X_test["MediaAnterior7"].to_numpy()

# Calcular métricas
def metricas_test(nombre, pred):
    return {
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred))
    }

resultados_test = pd.DataFrame([
    metricas_test("Media de 7 jornadas", pred_test_base),
    metricas_test("Random Forest", pred_test_rf)
])

print("RESULTADOS FINALES EN TEST:")
display(resultados_test.round(2))

# Mejora respecto al modelo de referencia
mae_base = resultados_test.iloc[0]["MAE"]
mae_rf = resultados_test.iloc[1]["MAE"]

mejora = (mae_base - mae_rf) / mae_base * 100

print(f"\nMejora del MAE: {mejora:.2f}%")

# Guardar resultados para GitHub
resultados_test.to_csv(
    "resultados_evaluacion_final.csv",
    index=False
)


## 17. Análisis de errores

Se comparan valores reales y predichos jornada por jornada, identificando picos y sesgos. Un aumento del error de Random Forest respecto a validación sugiere limitaciones de generalización para este período.


In [ ]:

# PASO 31: ANALISIS FINAL DE PREDICCIONES

import pandas as pd
import matplotlib.pyplot as plt

# Fechas correspondientes al conjunto de prueba
fechas_test = modelo_df.loc[X_test.index, "Fecha"]

# Construimos una tabla de comparación
analisis_test = pd.DataFrame({
    "Fecha": fechas_test.to_numpy(),
    "Real": y_test.to_numpy(),
    "Media7": pred_test_base,
    "RandomForest": pred_test_rf
})

# Calculamos errores absolutos
analisis_test["ErrorMedia7"] = abs(
    analisis_test["Real"] - analisis_test["Media7"]
)

analisis_test["ErrorRF"] = abs(
    analisis_test["Real"] - analisis_test["RandomForest"]
)

# Visualizamos las predicciones
plt.figure(figsize=(12, 5))

plt.plot(analisis_test["Fecha"],
         analisis_test["Real"],
         label="Ingresos reales")

plt.plot(analisis_test["Fecha"],
         analisis_test["Media7"],
         label="Media 7 jornadas")

plt.plot(analisis_test["Fecha"],
         analisis_test["RandomForest"],
         label="Random Forest")

plt.title("Comparación de predicciones - Periodo de prueba")
plt.xlabel("Fecha")
plt.ylabel("Ingresos comerciales (£)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("comparacion_predicciones_test.png", dpi=150)
plt.show()

# Días con los mayores errores de Random Forest
print("CINCO MAYORES ERRORES DE RANDOM FOREST:")
display(
    analisis_test.nlargest(5, "ErrorRF").round(2)
)

# Guardamos resultados reproducibles
analisis_test.to_csv(
    "predicciones_test.csv",
    index=False
)


## 18. Interpretación empresarial

Se cuantifican desviaciones por exceso y defecto del pronóstico, útiles para la planificación. **No representan pérdidas monetarias reales**, sino diferencias de estimación; su traducción a coste necesitaría supuestos adicionales.


In [ ]:

# PASO 32: METRICAS DE NEGOCIO

import numpy as np
import pandas as pd

# Utilizamos los resultados del paso 31
negocio = analisis_test.copy()

# Calculamos diferencias entre ingresos reales y predicciones
for modelo in ["Media7", "RandomForest"]:

    diferencia = negocio["Real"] - negocio[modelo]

    negocio[f"Subestimacion_{modelo}"] = (
        diferencia.clip(lower=0)
    )

    negocio[f"Sobreestimacion_{modelo}"] = (
        (-diferencia).clip(lower=0)
    )

# Resumimos los indicadores
resumen_negocio = []

for modelo in ["Media7", "RandomForest"]:

    subestimacion = negocio[f"Subestimacion_{modelo}"]
    sobreestimacion = negocio[f"Sobreestimacion_{modelo}"]

    resumen_negocio.append({
        "Modelo": modelo,
        "Subestimacion_total": round(subestimacion.sum(), 2),
        "Sobreestimacion_total": round(sobreestimacion.sum(), 2),
        "Dias_subestimados": int((subestimacion > 0).sum()),
        "Tasa_subestimacion_%": round(
            (subestimacion > 0).mean() * 100, 2
        )
    })

resumen_negocio = pd.DataFrame(resumen_negocio)

print("INDICADORES EMPRESARIALES:")
display(resumen_negocio)

print("\nNUMERO DE DIAS EVALUADOS:", len(negocio))


## 19. Exportación y visualización con Tableau

Se generan series históricas y pronósticos de prueba. Para Tableau Desktop configurado en español se crean además CSV con `;` como separador, coma decimal y codificación `utf-8-sig`. Los pasos de descarga mediante `google.colab.files` funcionan **solo en Google Colab**; si se ejecuta fuera de Colab, conviene omitir esas celdas y utilizar los CSV generados.


In [ ]:

# PASO 34: EXPORTAR DATOS PARA EL DASHBOARD

from pathlib import Path
import pandas as pd

# Creamos una carpeta para los resultados
carpeta = Path("dashboard_tfm")
carpeta.mkdir(exist_ok=True)

# 1. Ingresos comerciales históricos
historico = serie_final[
    ["Fecha", "IngresoComercial"]
].copy()

historico.to_csv(
    carpeta / "ingresos_historicos.csv",
    index=False
)

# 2. Predicciones del conjunto de prueba
predicciones = analisis_test[
    ["Fecha", "Real", "Media7", "RandomForest",
     "ErrorMedia7", "ErrorRF"]
].copy()

predicciones.to_csv(
    carpeta / "predicciones_test.csv",
    index=False
)

# Verificamos los archivos creados
for archivo in carpeta.glob("*.csv"):
    print(
        archivo.name,
        "-",
        round(archivo.stat().st_size / 1024, 2),
        "KB"
    )

print("\nFILAS EXPORTADAS:")
print("Histórico:", len(historico))
print("Predicciones:", len(predicciones))

print("\nPRIMERAS PREDICCIONES:")
display(predicciones.head())


In [ ]:

# PASO 35: DESCARGAR ARCHIVOS PARA TABLEAU

from google.colab import files

files.download(
    "dashboard_tfm/ingresos_historicos.csv"
)

files.download(
    "dashboard_tfm/predicciones_test.csv"
)


In [ ]:

# Exportar CSV compatible con configuración española

historico.to_csv(
    "ingresos_tableau.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

from google.colab import files
files.download("ingresos_tableau.csv")


In [ ]:

# PASO 38: EXPORTAR PREDICCIONES PARA TABLEAU

import pandas as pd
from google.colab import files

# Recuperamos el CSV del paso 34
predicciones = pd.read_csv(
    "dashboard_tfm/predicciones_test.csv"
)

# Exportamos en formato compatible con Tableau
predicciones.to_csv(
    "predicciones_tableau.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

# Verificamos los datos
print("Número de registros:", len(predicciones))
print("Columnas:", predicciones.columns.tolist())

display(predicciones.head())

# Descargamos el CSV
files.download("predicciones_tableau.csv")


## 20. Limitaciones y conclusión del experimento

La referencia de siete jornadas logra menor MAE en prueba que Random Forest, pese al mejor desempeño de este último en validación. Esto respalda **utilizar provisionalmente la media de siete jornadas** para este escenario concreto, supervisando MAE, RMSE, sesgo, evolución temporal y disponibilidad del dato. El alcance se limita al dataset y al protocolo descritos: no establece precisión futura garantizada.

**Despliegue conceptual:** al cierre de cada jornada con registro, actualizar las siete observaciones anteriores, emitir la predicción de la siguiente jornada registrada y guardar fecha, predicción, dato observado y error al conocerse. Revisar sesgos y cambios de comportamiento antes de modificar el procedimiento.

**Reproducibilidad:** ejecutar secuencialmente desde el comienzo; comprobar la existencia de `Online Retail.xlsx`, librerías y entorno Colab para las descargas. Conservar el notebook original y los registros de decisiones en la bitácora.
